In [1]:
from dotenv import load_dotenv

from typed_evals import evaluate

load_dotenv()

True

# Response Evaluation

The explicit `"response"` preset checks answer relevance and completeness. Omitting both `preset` and `metrics` checks answer relevance alone.

In [2]:
result = evaluate(
    input="What is the refund period?",
    response="You can request a refund within 30 days.",
    preset="response",  # response, rag, agent, None (default)
)

In [3]:
print("Overall Result:", result.passed)
for name, metric in result.metrics.items():
    print(f"Metric Name: {name}, Score: {metric.score}, Passed: {metric.passed}")

Overall Result: True
Metric Name: answer_relevancy, Score: 0.99, Passed: True
Metric Name: answer_completeness, Score: 0.99, Passed: True


# Use custom model

#### Openrouter

In [4]:
import os

from typed_evals.backends import JevBackend

In [11]:
openrouter_backend = JevBackend(
    model="microsoft/microsoft-decision-1",
    base_url="https://openrouter.ai/api",
    api_key=os.getenv("OPENROUTER_API_KEY"),
)

In [12]:
result = evaluate(
    input="What is the refund period?",
    response="You can request a refund within 30 days.",
    preset="response",
    backend=openrouter_backend,
)

In [13]:
print("Overall Result:", result.passed)
for name, metric in result.metrics.items():
    print(f"Metric Name: {name}, Score: {metric.score}, Passed: {metric.passed}")

Overall Result: True
Metric Name: answer_relevancy, Score: 0.9995121427091537, Passed: True
Metric Name: answer_completeness, Score: 0.998299277057694, Passed: True


#### HuggingFace

This example uses laya model from huggingface.

In [ ]:
# pip install "laya[serve]"

# USE_TF=0 LAYA_HOST=127.0.0.1 LAYA_MODELS=typed-decisions laya-serve

In [14]:
import os

from typed_evals.backends import SystemOneBackend

In [15]:
systemone_backend = SystemOneBackend(
    model="convaiinnovations/laya-typed-decisions",
    base_url="http://127.0.0.1:8000",
    api_key="api-key",
)

In [16]:
result = evaluate(
    input="What is the refund period?",
    response="You can request a refund within 30 days.",
    preset="response",
    backend=systemone_backend,
)

In [17]:
print("Overall Result:", result.passed)
for name, metric in result.metrics.items():
    print(f"Metric Name: {name}, Score: {metric.score}, Passed: {metric.passed}")

Overall Result: True
Metric Name: answer_relevancy, Score: 0.6086, Passed: True
Metric Name: answer_completeness, Score: 0.5673, Passed: True


# RAG Evaluation

The `"rag"` preset checks faithfulness, answer relevance, answer completeness, context relevance, and context sufficiency. `ContextRelevance` scores the fraction of source sentence/line units whose Noul probability meets `unit_threshold` (`0.5` by default). Its verdicts are derived booleans, without generated explanations. A relevant sentence among ten scores `0.1`; on-topic fabricated text can still be relevant. Omitted contexts are missing evidence; `contexts=[]` records known-empty retrieval.


In [18]:
result = evaluate(
    input="What is the refund period?",
    response="You can request a refund within 30 days.",
    contexts=["Refunds are allowed within 30 days of purchase."],
    preset="rag",  # response, rag, agent, None (default)
)

In [19]:
print("Overall Result:", result.passed)
for name, metric in result.metrics.items():
    print(f"Metric Name: {name}, Score: {metric.score}, Passed: {metric.passed}")

Overall Result: True
Metric Name: faithfulness, Score: 0.94, Passed: True
Metric Name: answer_relevancy, Score: 0.99, Passed: True
Metric Name: answer_completeness, Score: 0.98, Passed: True
Metric Name: context_relevance, Score: 1.0, Passed: True
Metric Name: context_sufficiency, Score: 0.98, Passed: True


# Agent Evaluation

The `"agent"` preset checks task completion, tool grounding, answer relevance, and answer completeness. A claimed action does not establish successful execution; the trace below records an actual failure.


In [20]:
result = evaluate(
    preset="agent",
    input="Create a support ticket for the export failure.",
    response="I created ticket T-42 for your export failure.",
    expected_outcome="A new support ticket exists for the reported export failure.",
    trace=[
        {
            "name": "create_ticket",
            "arguments": {"title": "Export failure"},
            "status": "error",
            "output": {"created": False},
            "error": "Permission denied",
        }
    ],
    # trace is a list of dictionaries representing the steps taken by the agent,
    # including the name of the step, the arguments passed, the status (success or error), the output, and any error messages.
)

In [21]:
print("Overall Result:", result.passed)
for name, metric in result.metrics.items():
    print(f"Metric Name: {name}, Score: {metric.score}, Passed: {metric.passed}")

Overall Result: False
Metric Name: task_completion, Score: 0.01, Passed: False
Metric Name: tool_grounding, Score: 0.02, Passed: False
Metric Name: answer_relevancy, Score: 0.62, Passed: True
Metric Name: answer_completeness, Score: 0.95, Passed: True


See more examples in the [examples folder](../examples)

For advanced usage, see notebook [here](advanced_usage.ipynb)